In [ ]:
#Import needed libraries
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [ ]:
#Load the data
city_df = pd.read_csv('city_day.csv')

# Drop rows with missing target (AQI)
city_df = city_df.dropna(subset=["AQI"])

# Drop non-useful columns
city_df = city_df.drop(columns=["City", "Date", "AQI_Bucket"], errors="ignore")


# Separate features and target
X = city_df.drop("AQI", axis=1)
y = city_df["AQI"]

In [ ]:
#Split the training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
#Fill the X-Values that are missing with the medians of that feature
train_medians = X_train.median()
X_train = X_train.fillna(train_medians)
X_test  = X_test.fillna(train_medians)

In [ ]:
#Scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
#Perform grid search with cross-validation to find the best decision Tree hyperparameters
parameter_grid = {
    "max_depth":         [3, 5, 8, 10, 15, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf":  [1, 2, 5],
    "criterion":         ["squared_error", "absolute_error"],
}

grid_search = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_grid=parameter_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train, y_train)

best_parameters = grid_search.best_params_

In [ ]:
#Train the model
decision_tree_model = DecisionTreeRegressor(**best_parameters, random_state=42)
decision_tree_model.fit(X_train, y_train)

In [ ]:
#Cross-validation stability check
cross_validation_r2   = cross_val_score(decision_tree_model, X_train, y_train, cv=5, scoring="r2")
cross_validation_rmse = cross_val_score(decision_tree_model, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error")
cross_validation_mae = cross_val_score(decision_tree_model, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")

print(f"\n5-Fold CV on Training Set:")
print(f"  MAE — Mean: {(-cross_validation_mae).mean():.4f} | Standard Deviation: {(-cross_validation_mae).std():.4f}")
print(f"  RMSE — Mean: {(-cross_validation_rmse).mean():.4f} | Standard Deviation: {(-cross_validation_rmse).std():.4f}")
print(f"  R²   — Mean: {cross_validation_r2.mean():.4f}  | Standard Deviation: {cross_validation_r2.std():.4f}")

In [ ]:
#Generate predictions on the test set, and then generate and display the evaluation metrics
y_prediction = decision_tree_model.predict(X_test)

mean_absolute_error  = mean_absolute_error(y_test, y_prediction)
root_mean_squared_error = root_mean_squared_error(y_test, y_prediction)
r2   = r2_score(y_test, y_prediction)

print(f"  MAE  (Mean Absolute Error):  {mean_absolute_error:.4f}")
print(f"  RMSE (Root Mean Squared Error): {root_mean_squared_error:.4f}")
print(f"  R²   (Coefficient of Determination): {r2:.4f}")

In [ ]:
#least features from greatest to least performance
importances = decision_tree_model.feature_importances_

feature_importance = pd.DataFrame({"Feature": X.columns,"Importances": importances,}).sort_values(by='Importances', ascending=False)

print(" Feature importance's: ")
print(feature_importance)
